# NLST pilot — validate slice numbers and timepoints

Runs entirely in Colab. Nothing to install locally.

Two questions:
1. Where does the T0/T1/T2 screening year live in the DICOM?
2. What does `sct_slice_num` in the `ctab` table actually index?

Run the cells in order. Cell 2 may answer question 1 on its own, before any images are downloaded.

## 1. Setup

In [ ]:
!pip install -q pydicom
import requests, zipfile, io, os, glob, json
import pandas as pd, numpy as np
import pydicom
import matplotlib.pyplot as plt

API = "https://nlst.cancerimagingarchive.net/nbia-api/services/v1"   # NLST has its own server
ROOT = "/content/nlst_pilot"
os.makedirs(ROOT, exist_ok=True)

PIDS = ["207375", "210581", "213754", "201095", "205350", "115962"]

# from ctab: the recorded nodule for each patient/exam
TARGETS = pd.DataFrame([
    dict(pid=207375, study_yr=0, slice=40,  long_mm=195, perp_mm=101, lobe="Right Upper"),
    dict(pid=210581, study_yr=2, slice=80,  long_mm=130, perp_mm=90,  lobe="Left Lower"),
    dict(pid=213754, study_yr=0, slice=144, long_mm=110, perp_mm=106, lobe="Left Upper"),
    dict(pid=201095, study_yr=2, slice=107, long_mm=107, perp_mm=390, lobe="Right Lower"),
    dict(pid=205350, study_yr=2, slice=285, long_mm=96,  perp_mm=7,   lobe="Left Lower"),
    dict(pid=115962, study_yr=1, slice=122, long_mm=90,  perp_mm=80,  lobe="Right Lower"),
])
TARGETS

## 2. What series exist, and what metadata do they carry?

This is metadata only — no pixels, so it runs in seconds. Look hard at `StudyDate` and
the description columns. If StudyDate varies within a patient, timepoints can be
derived by sorting. If every date is identical, we need a different field.

In [ ]:
rows = []
for pid in PIDS:
    r = requests.get(f"{API}/getSeries", params={"Collection": "NLST", "PatientID": pid}, timeout=60)
    r.raise_for_status()
    for s in r.json():
        rows.append(s)

series = pd.DataFrame(rows)
print("series found:", len(series))
print("\ncolumns returned by the API:")
print(list(series.columns))

cols = [c for c in ["PatientID","StudyInstanceUID","StudyDate","SeriesInstanceUID",
                    "SeriesDescription","SeriesNumber","ImageCount","BodyPartExamined",
                    "SliceThickness","ProtocolName","Manufacturer"] if c in series.columns]
series[cols].sort_values(["PatientID","StudyDate","SeriesNumber"] if "StudyDate" in cols else ["PatientID"])

In [ ]:
# Does StudyDate distinguish the three screens?
if "StudyDate" in series.columns:
    for pid, g in series.groupby("PatientID"):
        dates = sorted(g.StudyDate.dropna().unique())
        print(pid, "->", g.StudyInstanceUID.nunique(), "studies,",
              len(dates), "distinct dates:", dates)
else:
    print("no StudyDate in the API response -- check the DICOM headers in cell 5")

## 3. Download the images

About 3 GB into Colab's local disk. Colab's connection is fast, so this is usually
a few minutes. Re-running skips anything already downloaded.

In [ ]:
def fetch_series(uid, dest):
    if os.path.isdir(dest) and glob.glob(os.path.join(dest, "*.dcm")):
        return "cached"
    r = requests.get(f"{API}/getImage", params={"SeriesInstanceUID": uid},
                     stream=True, timeout=600)
    r.raise_for_status()
    buf = io.BytesIO(r.content)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(buf) as z:
        z.extractall(dest)
    return "ok"

for i, s in series.iterrows():
    pid = s["PatientID"]
    uid = s["SeriesInstanceUID"]
    dest = os.path.join(ROOT, str(pid), uid[-12:])
    try:
        status = fetch_series(uid, dest)
    except Exception as e:
        status = f"FAILED {type(e).__name__}"
    print(f"[{i+1}/{len(series)}] {pid} {uid[-12:]} {status}", flush=True)

!du -sh /content/nlst_pilot

## 4. Question 1 — where is the screening year?

Prints every candidate tag from one file per study. Whichever field differs
between a patient's studies is the timepoint.

In [ ]:
CANDIDATES = ["StudyDate","SeriesDate","AcquisitionDate","StudyDescription",
              "SeriesDescription","StudyComments","ImageComments","PatientComments",
              "ClinicalTrialTimePointID","ClinicalTrialTimePointDescription",
              "ClinicalTrialSeriesID","ClinicalTrialProtocolID","StudyID",
              "AccessionNumber","ContentDate"]

recs = []
for pid_dir in sorted(glob.glob(os.path.join(ROOT, "*"))):
    for ser_dir in sorted(glob.glob(os.path.join(pid_dir, "*"))):
        files = sorted(glob.glob(os.path.join(ser_dir, "*.dcm")))
        if not files:
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        rec = {"pid": os.path.basename(pid_dir),
               "series": os.path.basename(ser_dir),
               "n_slices": len(files),
               "StudyInstanceUID": str(getattr(ds, "StudyInstanceUID", ""))[-10:]}
        for t in CANDIDATES:
            v = getattr(ds, t, None)
            if v not in (None, ""):
                rec[t] = str(v)
        recs.append(rec)

tags = pd.DataFrame(recs)
pd.set_option("display.max_columns", None, "display.width", 250)
print("tags that are populated somewhere:")
print([c for c in tags.columns if tags[c].notna().any()])
tags

In [ ]:
# Which populated tag actually varies between studies of the same patient?
for c in tags.columns:
    if c in ("pid","series","n_slices"):
        continue
    varies = tags.groupby("pid")[c].nunique(dropna=True)
    if (varies > 1).any():
        print(f"{c}: varies within patient -- candidate timepoint field")
        print(tags.groupby('pid')[c].unique().to_string(), "\n")

## 5. Question 2 — what does sct_slice_num index?

For each recorded nodule, every series in that patient is shown under five competing
interpretations of the slice number. Find the tumour. Note the column heading.

In [ ]:
LUNG_LEVEL, LUNG_WIDTH = -600, 1500

def load_series(d):
    out = []
    for p in glob.glob(os.path.join(d, "*.dcm")):
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            continue
        z = None
        if hasattr(ds, "ImagePositionPatient"):
            z = float(ds.ImagePositionPatient[2])
        elif hasattr(ds, "SliceLocation"):
            z = float(ds.SliceLocation)
        out.append((z, int(getattr(ds, "InstanceNumber", 0)), ds))
    if not out:
        return [], []
    out.sort(key=lambda t: (-t[0] if t[0] is not None else t[1]))   # superior first
    return [o[2] for o in out], [o[1] for o in out]

def windowed(ds):
    a = ds.pixel_array.astype(np.float32)
    hu = a * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    lo, hi = LUNG_LEVEL - LUNG_WIDTH/2, LUNG_LEVEL + LUNG_WIDTH/2
    return np.clip((hu - lo)/(hi - lo), 0, 1)

def rules(i, n, inums):
    r = {"1-based top-down": i-1, "0-based top-down": i,
         "1-based bottom-up": n-i, "0-based bottom-up": n-i-1}
    r = {k: (v if 0 <= v < n else None) for k, v in r.items()}
    m = [j for j, x in enumerate(inums) if x == i]
    r["InstanceNumber tag"] = m[0] if m else None
    return r

for _, t in TARGETS.iterrows():
    pdir = os.path.join(ROOT, str(t.pid))
    sers = [d for d in sorted(glob.glob(os.path.join(pdir, "*"))) if glob.glob(os.path.join(d, "*.dcm"))]
    sers = [(d, *load_series(d)) for d in sers]
    sers = [(d, o, n) for d, o, n in sers if len(o) >= 20]
    if not sers:
        print("no usable series for", t.pid); continue

    names = list(rules(int(t.slice), len(sers[0][1]), sers[0][2]).keys())
    fig, ax = plt.subplots(len(sers), len(names),
                           figsize=(2.6*len(names), 2.6*len(sers)), squeeze=False)
    for r, (d, ordered, inums) in enumerate(sers):
        cand = rules(int(t.slice), len(ordered), inums)
        for c, nm in enumerate(names):
            a = ax[r][c]; a.set_xticks([]); a.set_yticks([])
            idx = cand[nm]
            if idx is None:
                a.text(.5,.5,"out of range",ha="center",va="center",transform=a.transAxes,fontsize=7)
            else:
                a.imshow(windowed(ordered[idx]), cmap="gray")
            if r == 0: a.set_title(nm, fontsize=8)
            if c == 0: a.set_ylabel(f"{os.path.basename(d)}\n{len(ordered)} sl", fontsize=6)
    fig.suptitle(f"pid {t.pid}  T{t.study_yr}  slice {int(t.slice)}  "
                 f"{t.long_mm:.0f}x{t.perp_mm:.0f}mm  {t.lobe}", fontsize=11)
    fig.tight_layout(); plt.show()

## 6. What to send back

- The table from cell 2 and the output of the "varies within patient" cell
- The images from cell 5, plus which column heading the tumour appeared under

If no column ever shows the tumour, say so — that is a valid answer and we label per exam instead.